# 🌍 Actividad: Pipeline de datos — dos fuentes → una tabla en NocoDB
**Extraer → Transformar → Cargar** con datos reales de internet.
- Fuente A: **Banco Mundial** (población por país, sin clave)
- Fuente B: **REST Countries v5** (nombre, capital, región y población, con tu API key gratuita)
- Destino: la tabla `Paises` de **NocoDB** (plan gratuito) configurada por el docente

Ejecutá las celdas **en orden**. Cuando veas `TU_TABLA_ID`, reemplazá ese valor.

## 0. Configuración y secretos 🔑

In [ ]:
# Antes de ejecutar: panel izquierdo -> pestaña 🔑 Secrets -> creá DOS secretos:
#   NOCODB_TOKEN      = tu API token de app.nocodb.com (Account Settings > Tokens)
#   RESTCOUNTRIES_KEY = tu API key de restcountries.com (registro gratuito)
# Además, pedile al docente el identificador de la tabla y pegalo acá:
TABLA_ID = "TU_TABLA_ID"   # ej: m6ps8w8t9k4abcd

try:
    from google.colab import userdata
    import os
    os.environ['NOCODB_TOKEN'] = userdata.get('NOCODB_TOKEN')
    os.environ['RC_KEY'] = userdata.get('RESTCOUNTRIES_KEY')
    print('✅ Secretos cargados')
except Exception as e:
    print(f'❌ Falta un secreto ({e}). Mismo nombre, sin espacios.')
print('Tabla destino:', TABLA_ID)


## 1. EXTRACT — Fuente A: Banco Mundial 🏦
Una sola llamada: `mrv=1` pide solo el **último año disponible** por país y `per_page=300` traelos ~265 registros completos en una página. La respuesta es `[metadatos, registros]`.

In [ ]:
import requests, json
import pandas as pd
URL_WB = "https://api.worldbank.org/v2/es/country/all/indicator/SP.POP.TOTL?format=json&per_page=300&mrv=1"
r = requests.get(URL_WB, timeout=30)
r.raise_for_status()   # ORO: un 500 con JSON "decodifica bien"; así lo detectamos
wb_resp = r.json()
meta, registros = wb_resp[0], wb_resp[1]
print(f"Indicador actualizado: {meta['lastupdated']} · registros: {meta['total']}")
df_wb_raw = pd.DataFrame(registros)
df_wb_raw.head()


## 2. TRANSFORM — limpieza de la fuente A
El Banco Mundial incluye **agrupaciones** (World, Latin America & Caribbean…) junto con lospaíses reales. Sus códigos ISO3 son agregados — los excluimos con la lista verificada de laAPI de países del propio Banco Mundial (`region.id == 'NA'`). Luego elegimos y renombramoscolumnas para que queden **iguales** a las de la fuente B.

In [ ]:
# Agregados del Banco Mundial con región 'NA' (verificado el 2026-10-02 sobre /v2/country)
AGG_CODES = {'AFE','AFR','AFW','ARB','BEA','BEC','BHI','BLA','BMN','BSS','CAA','CEA','CEB',
             'CEU','CLA','CME','CSA','CSS','DEA','DEC','DLA','DMN','DNS','DSA','DSF','DSS',
             'EAP','EAR','EAS','ECA','ECS','EMU','EUU','FXS','HIC','HPC','IBB','IBD','IBT',
             'IDA','IDB','IDX','INX','LAC','LCN','LDC','LIC','LMC','LMY','LTE','MDE','MEA',
             'MIC','MNA','NAC','NAF','NRS','NXS','OED','OSS','PRE','PSS','PST','RRS','SAS',
             'SSA','SSF','SST','SXZ','TEA','TEC','TLA','TMN','TSA','TSS','UMC','WLD','XZN'}
df_wb = df_wb_raw[['countryiso3code','country','date','value']].copy()
df_wb.columns = ['codigo_iso3','nombre_pais','anio','poblacion']
df_wb['nombre_pais'] = df_wb['nombre_pais'].apply(lambda x: x['value'])   # era dict {id, value}
df_wb['anio'] = df_wb['anio'].astype(int)
df_wb = df_wb[df_wb['codigo_iso3'] != '']            # agregados sin código (ej. "Ingreso alto")
df_wb = df_wb[~df_wb['codigo_iso3'].isin(AGG_CODES)].dropna(subset=['poblacion'])
df_wb['fuente'] = 'Banco Mundial'
df_wb = df_wb.reset_index(drop=True)
print(f"Países reales del Banco Mundial: {len(df_wb)}")
df_wb.head()


## 3. EXTRACT — Fuente B: REST Countries v5 🏳️
Requiere tu clave en el header `Authorization: Bearer`. El plan gratuito pagina a `limit=1–100`,así que los 249 países salen en 3 llamadas (offset 0, 100, 200). La respuesta viaja en`data.objects` y los campos clave verificados son `names.common`, `codes.alpha_3`,`capitals[0].name`, `region` y `population`.

In [ ]:
RC_KEY = os.environ['RC_KEY']
URL_RC = "https://api.restcountries.com/countries/v5"
HEADERS_RC = {"Authorization": f"Bearer {RC_KEY}"}
paises_rc = []
for offset in (0, 100, 200):
    r = requests.get(URL_RC, headers=HEADERS_RC, params={'limit': 100, 'offset': offset}, timeout=30)
    r.raise_for_status()
    pagina = r.json()['data']['objects']
    print(f"Página offset={offset}: {len(pagina)} países")
    paises_rc.extend(pagina)
    if len(pagina) < 100 and offset >= 200:
        break
print(f"Total REST Countries: {len(paises_rc)}")


## 4. TRANSFORM — fuente B con el MISMO formato de fila

In [ ]:
filas_rc = [{
    'codigo_iso3': c['codes']['alpha_3'],
    'nombre_pais': c['names']['common'],
    'anio': 2026,   # la population de v5 es "al momento"; la anotamos con el año de ejecución
    'poblacion': c.get('population'),
    'fuente': 'REST Countries',
} for c in paises_rc]
df_rc = pd.DataFrame(filas_rc).dropna(subset=['poblacion'])
print(f"Países de REST Countries: {len(df_rc)}")
df_rc.head()


## 5. Carga ⬆️ — ambas fuentes hacia LA MISMA tabla
Ambas fuentes ya tienen los MISMOS cinco campos: `codigo_iso3, nombre_pais, anio, poblacion, fuente`.El endpoint de NocoDB acepta **un array completo en una sola llamada** (bulk insert).

In [ ]:
df_total = pd.concat([df_wb, df_rc], ignore_index=True)
print(df_total['fuente'].value_counts())
df_total.head(10)


In [ ]:
NOCODB_TOKEN = os.environ['NOCODB_TOKEN']
url = f"https://app.nocodb.com/api/v2/tables/{TABLA_ID}/records"
headers = {'xc-token': NOCODB_TOKEN}
filas = df_total.to_dict(orient='records')

# NocoDB cloud acepta un maximo de 100 filas por llamada (ERR_MAX_PAYLOAD_LIMIT_EXCEEDED)
TAMANO_TANDA = 100
tandas = [filas[i:i+TAMANO_TANDA] for i in range(0, len(filas), TAMANO_TANDA)]
print(f"Enviando {len(filas)} filas en {len(tandas)} tandas de hasta {TAMANO_TANDA}...")

ok, errores = 0, []
for i, tanda in enumerate(tandas, 1):
    r = requests.post(url, json=tanda, headers=headers, timeout=60)
    print(f"Tanda {i}/{len(tandas)}: HTTP {r.status_code}")
    if 200 <= r.status_code < 300:
        ok += len(tanda)
    else:
        errores.append((i, r.status_code, r.text[:200]))

print(f"Filas insertadas: {ok}/{len(filas)}")
if errores:
    print('Revisa estos errores:', errores[:3])
print(df_total.tail(3))

## 6. Verificación y discusión 🤔
1. Abrí tu tabla en `app.nocodb.com`: ¿cuántas filas hay? ¿de dónde salió cada una?
2. Filtrá un país (ej. `ARG`) y compará la población según ambas fuentes. ¿Coinciden? ¿Por qué casi nunca al 100%?
3. Si re-ejecutás todo con la tabla llena, ¿qué pasa con los 1.000 registros del plan gratuito? **Vacía la tabla antes de repetir.**
4. Ojo con el error `422`: significa que tus columnas no coinciden con los campos de la tabla.
5. `401` → token mal copiado/expirado · `429` → demasiadas llamadas seguidas, esperá unos segundos.